# Recuperando publicações do Brasil sobre doenças negligenciadas através da API da PubMed

## ⚠️ Limitações conhecidas desta extração

Leia antes de usar os dados. Estes números **não** representam toda a produção brasileira sobre DTNs.

**1. Cobertura da base**
- Só inclui o que está indexado na **PubMed**. Revistas brasileiras indexadas apenas na LILACS/SciELO ficam de fora.

**2. Filtro de país (`Brazil[Affiliation] OR Brasil[Affiliation]`)**
- **Antes de 2014** a PubMed registrava só a afiliação do **primeiro autor**: artigos com brasileiros apenas como coautores não são recuperados nesse período. Comparações de tendência ao longo do tempo ficam enviesadas (parte do "crescimento" pós-2014 pode ser artefato).
- Afiliações que não citam o país (ex.: "Instituto Evandro Chagas, Ananindeua, PA") e registros sem afiliação não são recuperados.

**3. Estratégia de busca das doenças**
- Alguns termos trazem **falsos positivos**: "Treponema pallidum" (sífilis), "Chlamydia trachomatis" (IST), "Staphylococcus aureus", "Streptococcus spp.", "Alphavirus", "Aedes Aegypti" (também Zika), "Gale", "Raiva". Os resultados precisam de triagem/filtragem antes de serem tratados como "artigos sobre DTNs".
- A busca não usa descritores MeSH explicitamente; artigos que só mencionam a doença no texto completo podem ficar de fora.

**4. Processamento**
- Lotes de download que falharam ficam em `pmids_falhos`. Se a lista não estiver vazia, o conjunto está incompleto.
- Artigos sem data viram `Ano = "Sem "` (o texto "Sem Data" cortado em 4 caracteres).
- Artigos recentes muitas vezes ainda não têm termos MeSH.

**5. Enriquecimento com a OpenAlex**
- Artigos que a OpenAlex não encontra ficam com citações e FWCI vazios (coluna `Encontrado_OpenAlex` = False). Ao calcular médias, lembre que esses artigos não entram.
- Instituições e estados dependem da identificação de afiliações feita pela OpenAlex, que às vezes falha (afiliação sem instituição ou instituição errada). Confira a lista de "sem estado" impressa no notebook e uma amostra de `brasil_dtns_autorias.csv`.
- Citações e FWCI mudam com o tempo; registre a data da extração.

**6. Reprodutibilidade**
- Se `brasil_dtns_pubmed_bruto.csv` ou `brasil_dtns.csv` já existirem, o notebook lê esses arquivos em vez de baixar de novo. Mudanças na query ou nas bases **não** são detectadas: apague os arquivos para atualizar os dados.
- A PubMed e a OpenAlex são atualizadas diariamente. Rodar o notebook em outra data dá outro resultado. Anote a data de execução junto com os arquivos gerados.


## Configuração inicial

In [1]:
#Rode a célula abaixo apenas se não tiver as bibliotecas instaladas
#!pip install pandas biopython openpyxl shapely

In [2]:
import pandas as pd
from Bio import Entrez
import requests
import time
import os


In [3]:
# ATENÇÃO: Substitua pelo seu e-mail real
Entrez.email = "seu_email@exemplo.com"

# ⚠️ Coloque a sua chave de API gerada no site da OpenAlex
api_key_openalex = "9F42v8YLDdW8s37xYQN3Pb"

# Arquivos gerados pelo notebook. Se já existirem, as etapas de download são puladas
# e os dados são lidos deles. Para baixar de novo, apague o arquivo correspondente.
ARQUIVO_PUBMED = 'brasil_dtns_pubmed_bruto.csv'   # PubMed (busca + metadados)
ARQUIVO_FINAL = 'brasil_dtns.csv'                 # PubMed + OpenAlex
ARQUIVO_AUTORIAS = 'brasil_dtns_autorias.csv'     # artigo x instituição, com estado
ARQUIVO_INSTITUICOES = 'brasil_instituicoes.csv'  # cadastro das instituições brasileiras


In [4]:
# ATENÇÃO: Substitua pelo seu e-mail real
Entrez.email = "seu_email@exemplo.com"

# ⚠️ Coloque a sua chave de API gerada no site da OpenAlex
api_key_openalex = "9F42v8YLDdW8s37xYQN3Pb"

# Arquivos gerados pelo notebook. Se já existirem, as etapas de download são puladas
# e os dados são lidos deles. Para baixar de novo, apague o arquivo correspondente.
ARQUIVO_PUBMED = 'brasil_dtns_pubmed_bruto.csv'   # PubMed (busca + metadados)
ARQUIVO_FINAL = 'brasil_dtns.csv'                 # PubMed + OpenAlex
ARQUIVO_AUTORIAS = 'brasil_dtns_autorias.csv'     # artigo x instituição, com estado
ARQUIVO_INSTITUICOES = 'brasil_instituicoes.csv'  # cadastro das instituições brasileiras


## Definindo estrtagégia de busca

In [5]:
# 1. Definindo a estratégia de busca (Doenças)
diseases_query = """(Ancilostomíase OR Ancylostomiasis OR Anquilostomiasis OR Ankylostomose OR "Necator americanus" OR 
"Ancylostoma duodenale" OR Ascaridíase OR Ascaridiasis OR "Infection à Ascaridia" OR "Ascaris spp." OR 
"Ascaris lumbricoides" OR Bouba OR Yaws OR Buba OR Pian OR "Treponema pallidum" OR "Vírus Chikungunya" OR 
"Chikungunya virus" OR "Virus Chikungunya" OR "Virus du chikungunya" OR Alphavirus OR Clonorquíase OR 
Clonorchiasis OR Clonorquiasis OR Clonorchiase OR "Clonorchis sinensis" OR Cromoblastomicose OR 
Chromoblastomycosis OR Cromoblastomicosis OR Chromoblastomycose OR Dengue OR Flavivírus OR "DENV-1" OR 
"DENV-2" OR "DENV-3" OR "DEN-V 4" OR "Aedes Aegypti" OR "Doença de Chagas" OR "Tripanossomíase americana" OR 
"Chagas Disease" OR "Enfermedad de Chagas" OR "Maladie de Chagas" OR "Trypanosoma cruzi" OR Dracunculíase OR 
Dracunculiasis OR Dracunculose OR "Dracunculus medinensis" OR Equinococose OR Hidatidose OR Echinococcosis OR 
Equinococosis OR Échinococcose OR "Echinococcus granulosus" OR "E. multilocularis" OR "Echinococcus multilocularis" OR 
Escabiose OR Scabies OR Escabiosis OR Gale OR "Sarcoptes scabiei" OR "Sarcoptes scabiei var. hominis" OR 
Esporotricose OR Sporotrichosis OR Esporotricosis OR Sporotrichose OR "Sporothrix brasiliensis" OR 
"Sporothrix schenckii" OR "Sporothrix globose" OR Esquistossomose OR Schistosomiasis OR Esquistosomiasis OR 
Schistosomiase OR Schistosoma OR Fasciolíase OR Fasciolase OR "Fasciola gigantica" OR "F. hepatica" OR 
"Fasciola hepatica" OR Filariose OR "Filariose linfática" OR Elefantíase OR Filariasis OR Filarioses OR 
"Wuchereria bancrofti" OR "Brugia timori" OR Hanseníase OR Leprosy OR Lepra OR Lèpre OR "Mycobacterium leprae" OR 
"bacilo de Hansen" OR "Leishmaniose Tegumentar Difusa" OR "Leishmaniasis, Diffuse Cutaneous" OR 
"Leishmaniasis Cutánea Difusa" OR "Leishmaniose cutanée diffuse" OR Leishmania OR "Leishmaniose Visceral" OR 
Calazar OR "Leishmaniose Cutânea" OR "Leishmaniasis, Cutaneous" OR "Leishmaniasis Cutánea" OR 
"Leishmaniose cutanée" OR "Leishmaniose Mucocutânea" OR "Leishmaniasis, Mucocutaneous" OR 
"Leishmaniasis Mucocutánea" OR "Leishmaniose cutanéomuqueuse" OR Micetoma OR Mycetoma OR Mycétome OR 
eumicetoma OR actinomicetoma OR Noma OR "Fusobacterium necrophorum" OR "Prevotella intermedia" OR 
"Borrelia vincentii" OR "Treponema denticola" OR "Staphylococcus aureus" OR "Streptococcus spp." OR 
Oncocercose OR Onchocerciasis OR Oncocercosis OR Onchocercose OR "Onchocerca volvulus" OR Opistorquíase OR 
Opisthorchiasis OR Opistorquiasis OR Opisthorchiase OR "Opisthorchis viverrine" OR "Opisthorchis felineus" OR 
Paragonimíase OR Paragonimiasis OR Paragonimose OR "Paragonimus westermani" OR "Paragonimus kellicotti" OR 
"Paragonimus heterotremus" OR "Picadas de cobra" OR "Acidente Ofídico" OR "Acidentes Ofídicos" OR "Snake Bites" OR 
"Mordeduras de Serpientes" OR "Morsures de serpente" OR Raiva OR Rabies OR Rabia OR "Rage (maladie)" OR 
"Vírus da Raiva" OR "Rabies vírus" OR "Virus de la Rabia" OR "Virus de la rage" OR "Vírus do gênero Lyssavirus" OR 
Rhabdoviridae OR Teníase OR Taeniasis OR Teniasis OR Taeniase OR "Taenia solium" OR "Taenia saginata" OR 
Cisticercose OR Cysticercosis OR Cisticercosis OR Cysticercose OR Tracoma OR Trachoma OR Trachome OR 
"Chlamydia trachomatis" OR Tricuríase OR Trichuriasis OR Tricuriasis OR Trichocéphalose OR "Trichuris trichiura" OR 
"Tripanossomíase Africana" OR "Trypanosomiasis, African" OR "Tripanosomiasis Africana" OR "Maladie du sommeil" OR 
"Trypanosoma brucei" OR "Trypanosoma brucei gambiense" OR "Trypanosoma brucei rhodesiense" OR Tungíase OR 
Tungiasis OR Tungose OR "Tunga penetrans" OR "Úlcera de Buruli" OR "Buruli ulcer" OR "Ulcère de Buruli" OR 
"Mycobacterium ulcerans")"""

# 2. Definindo a restrição pelo país (Brasil)
# O campo [Affiliation] é texto livre; quase toda afiliação termina com o país.
# Atenção: antes de 2014 a PubMed registrava apenas a afiliação do primeiro autor.
brasil_query = """(Brazil[Affiliation] OR Brasil[Affiliation])"""

# Combinando as duas buscas
full_query = f"{diseases_query} AND {brasil_query}"

print(full_query)

(Ancilostomíase OR Ancylostomiasis OR Anquilostomiasis OR Ankylostomose OR "Necator americanus" OR 
"Ancylostoma duodenale" OR Ascaridíase OR Ascaridiasis OR "Infection à Ascaridia" OR "Ascaris spp." OR 
"Ascaris lumbricoides" OR Bouba OR Yaws OR Buba OR Pian OR "Treponema pallidum" OR "Vírus Chikungunya" OR 
"Chikungunya virus" OR "Virus Chikungunya" OR "Virus du chikungunya" OR Alphavirus OR Clonorquíase OR 
Clonorchiasis OR Clonorquiasis OR Clonorchiase OR "Clonorchis sinensis" OR Cromoblastomicose OR 
Chromoblastomycosis OR Cromoblastomicosis OR Chromoblastomycose OR Dengue OR Flavivírus OR "DENV-1" OR 
"DENV-2" OR "DENV-3" OR "DEN-V 4" OR "Aedes Aegypti" OR "Doença de Chagas" OR "Tripanossomíase americana" OR 
"Chagas Disease" OR "Enfermedad de Chagas" OR "Maladie de Chagas" OR "Trypanosoma cruzi" OR Dracunculíase OR 
Dracunculiasis OR Dracunculose OR "Dracunculus medinensis" OR Equinococose OR Hidatidose OR Echinococcosis OR 
Equinococosis OR Échinococcose OR "Echinococcus granul

## Recuperando publicações através da API

O ESearch da PubMed devolve no máximo 10.000 PMIDs por consulta. Como o escopo Brasil ultrapassa esse limite, a busca é feita ano a ano (e mês a mês, caso algum ano passe de 9.999 resultados). O download dos metadados também é feito em lotes.

In [6]:
if os.path.exists(ARQUIVO_PUBMED):
    print(f"{ARQUIVO_PUBMED} já existe: busca na PubMed pulada (apague o arquivo para buscar de novo).")
else:
    import datetime

    LIMITE_ESEARCH = 9999

    def buscar_pmids(termo, inicio, fim):
        handle = Entrez.esearch(db="pubmed", term=termo, mindate=inicio, maxdate=fim,
                                datetype="pdat", retmax=LIMITE_ESEARCH)
        resultado = Entrez.read(handle)
        handle.close()
        time.sleep(0.4)  # NCBI permite 3 requisições/s sem chave de API
        return int(resultado["Count"]), resultado["IdList"]

    print("Buscando PMIDs na PubMed (ano a ano)...")
    pmids = []
    ano_atual = datetime.date.today().year

    for ano in range(1900, ano_atual + 1):
        total, ids = buscar_pmids(full_query, str(ano), str(ano))
        if total > LIMITE_ESEARCH:
            # Ano grande demais para uma única consulta: dividimos por mês
            ids = []
            for mes in range(1, 13):
                total_mes, ids_mes = buscar_pmids(full_query, f"{ano}/{mes:02d}", f"{ano}/{mes:02d}")
                if total_mes > LIMITE_ESEARCH:
                    print(f"⚠️ {ano}/{mes:02d}: {total_mes} resultados, apenas {LIMITE_ESEARCH} recuperados")
                ids += ids_mes
        if total:
            print(f"{ano}: {total} artigos")
        pmids += ids

    # Um mesmo artigo pode aparecer em dois anos (data impressa vs. eletrônica)
    pmids = list(dict.fromkeys(pmids))


brasil_dtns_pubmed_bruto.csv já existe: busca na PubMed pulada (apague o arquivo para buscar de novo).


In [7]:
#print(pmids)

In [8]:
pmids_falhos = []

if os.path.exists(ARQUIVO_PUBMED):
    papers = None
    print(f"{ARQUIVO_PUBMED} já existe: download dos metadados pulado.")
else:
    print(f"Foram encontrados {len(pmids)} artigos. Baixando metadados...")

    tamanho_lote_pubmed = 200

    def baixar_lote(lote, tentativas=3):
        for tentativa in range(tentativas):
            try:
                fetch_handle = Entrez.efetch(db="pubmed", id=lote, retmode="xml")
                resposta = Entrez.read(fetch_handle)
                fetch_handle.close()
                return resposta["PubmedArticle"]
            except Exception as e:
                print(f"Erro (tentativa {tentativa + 1}/{tentativas}, {len(lote)} PMIDs): {e}")
                time.sleep(5 * (tentativa + 1))
        return None

    if pmids:
        papers = {"PubmedArticle": []}
        for i in range(0, len(pmids), tamanho_lote_pubmed):
            lote = pmids[i:i + tamanho_lote_pubmed]
            artigos = baixar_lote(lote)
            if artigos is None:
                # Lote falhou em todas as tentativas: dividimos em lotes menores
                # para isolar eventuais PMIDs problemáticos
                print(f"Lote {i // tamanho_lote_pubmed + 1} falhou; tentando em lotes de 20...")
                for j in range(0, len(lote), 20):
                    sublote = lote[j:j + 20]
                    artigos_sub = baixar_lote(sublote)
                    if artigos_sub is None:
                        pmids_falhos += sublote
                    else:
                        papers["PubmedArticle"] += artigos_sub
            else:
                papers["PubmedArticle"] += artigos
            time.sleep(0.4)
            print(f"{min(i + tamanho_lote_pubmed, len(pmids))}/{len(pmids)} processados", end="\r")
        print("\nDownload concluído! Os dados brutos estão salvos na variável 'papers'.")
        if pmids_falhos:
            print(f"⚠️ {len(pmids_falhos)} PMIDs não puderam ser baixados (lista em 'pmids_falhos'). "
                  "Rode esta célula novamente com pmids = pmids_falhos para tentar de novo.")
    else:
        papers = None
        print("Nenhum artigo encontrado com essa estratégia.")


brasil_dtns_pubmed_bruto.csv já existe: download dos metadados pulado.


In [9]:
##Análise do que vem na resposta da API do pubmed

# Função recursiva que "mergulha" nos dados aninhados
def explorar_aninhamento(dado, nivel=0):
    espaco = "    " * nivel # Cria os espaços de indentação baseados na profundidade
    
    # 1. Se for um Dicionário (gavetas com nomes, ex: 'Article')
    if isinstance(dado, dict) or hasattr(dado, 'items'):
        if not dado: # Checa se o dicionário está vazio (ex: {})
            print(f"{espaco}  ➔ [Dicionário Vazio]")
        else:
            for chave, valor in dado.items():
                print(f"{espaco}🔹 {chave}:")
                explorar_aninhamento(valor, nivel + 1)
            
    # 2. Se for uma Lista (ex: 'AuthorList', onde há vários autores)
    elif isinstance(dado, list):
        if not dado: # Checa se a lista está vazia (ex: [])
            print(f"{espaco}  ➔ [Lista Vazia]")
        else:
            for index, item in enumerate(dado):
                print(f"{espaco}  [{index}]") # Mostra a posição na lista (0, 1, 2...)
                explorar_aninhamento(item, nivel + 1)
            
    # 3. Se for o valor final (o texto em si, número, etc.)
    else:
        # Lidando com o valor nulo (None) ou strings vazias ("")
        # Convertendo para string para garantir que pegamos espaços em branco também
        dado_str = str(dado).strip() if dado is not None else ""
        
        if not dado_str:
            print(f"{espaco}  ➔ [Vazio / Null]")
        else:
            # Imprime o valor normalmente
            print(f"{espaco}  ➔ {dado}") 

# -------------------------------------------------------------------
# EXECUÇÃO DO CÓDIGO
# -------------------------------------------------------------------

# Verifica se conseguimos baixar algo
if papers and 'PubmedArticle' in papers and len(papers['PubmedArticle']) > 0:
    
    primeiro_artigo = papers['PubmedArticle'][0]
    
    print("=====================================================")
    print("        EXPLORANDO OS DADOS DO PRIMEIRO ARTIGO       ")
    print("=====================================================\n")
    
    # Percorre os blocos principais para manter a divisão por linhas (---)
    for chave_principal, conteudo_principal in primeiro_artigo.items():
        print(f"🟩 BLOCO PRINCIPAL: {chave_principal}\n")
        
        # Chamamos nossa função recursiva atualizada!
        explorar_aninhamento(conteudo_principal, nivel=1)
        
        print("\n" + "=" * 50 + "\n")
            
else:
    print("Nenhum dado para explorar.")

Nenhum dado para explorar.


## Extraindo resposta da API para um dataframe

In [10]:
## Extraindo os dados para um dataframe

data = []

pubmed_lido_do_arquivo = os.path.exists(ARQUIVO_PUBMED)

if pubmed_lido_do_arquivo:
    df = pd.read_csv(ARQUIVO_PUBMED, dtype=str, keep_default_na=False)
    print(f"{len(df)} artigos lidos de {ARQUIVO_PUBMED}.")
    display(df.head())

elif papers and 'PubmedArticle' in papers:
    print(f"Processando {len(papers['PubmedArticle'])} artigos...")
    
    for article in papers['PubmedArticle']:
        medline = article['MedlineCitation']
        article_data = medline['Article']
        
        # 1. Identificadores Básicos
        pmid = str(medline.get('PMID', ''))
        title = str(article_data.get('ArticleTitle', ''))
        journal = str(article_data.get('Journal', {}).get('Title', ''))
        
        # [NOVO] País da Revista
        journal_country = str(medline.get('MedlineJournalInfo', {}).get('Country', ''))
        
        # [NOVO] Idioma do Artigo
        # Às vezes o artigo é publicado em mais de uma língua (ex: eng, por)
        languages = article_data.get('Language', [])
        language_str = "; ".join([str(lang) for lang in languages])
        
        # 2. DOI
        doi = ""
        if 'PubmedData' in article and 'ArticleIdList' in article['PubmedData']:
            for pmid_data in article['PubmedData']['ArticleIdList']:
                if pmid_data.attributes.get('IdType') == 'doi':
                    doi = str(pmid_data)
                    break
        
        # 3. Ano de Publicação
        year = "Sem Data"
        if 'Journal' in article_data and 'JournalIssue' in article_data['Journal']:
            pub_date = article_data['Journal']['JournalIssue'].get('PubDate', {})
            year = pub_date.get('Year', pub_date.get('MedlineDate', 'Sem Data'))
            
        # 4. Tipo de Publicação
        pub_types = []
        if 'PublicationTypeList' in article_data:
            for pt in article_data['PublicationTypeList']:
                pub_types.append(str(pt))
        pub_types_str = "; ".join(pub_types)
        
        # 5. Autores e Afiliações
        authors_curto = []
        authors_completo = [] # [NOVO] Lista para nomes completos
        affiliations_set = set()
        first_author = ""
        
        if 'AuthorList' in article_data:
            for i, author in enumerate(article_data['AuthorList']):
                
                # Pegando as partes do nome
                last_name = author.get('LastName', '')
                initials = author.get('Initials', '')
                fore_name = author.get('ForeName', '') # [NOVO] Nome de batismo
                
                # Lógica para o nome Curto (Ex: Silva J)
                if last_name and initials:
                    nome_curto = f"{last_name} {initials}"
                elif 'CollectiveName' in author:
                    nome_curto = str(author['CollectiveName'])
                else:
                    nome_curto = "Desconhecido"
                    
                # Lógica para o nome Completo (Ex: João Silva)
                if fore_name and last_name:
                    nome_completo = f"{last_name}, {fore_name}"
                elif 'CollectiveName' in author:
                    nome_completo = str(author['CollectiveName'])
                else:
                    nome_completo = nome_curto # Fallback se não tiver fore_name
                    
                # Adicionando nas listas
                authors_curto.append(nome_curto)
                authors_completo.append(nome_completo)
                
                if i == 0:
                    first_author = nome_curto
                
                # Afiliações desse autor
                if 'AffiliationInfo' in author:
                    for aff in author['AffiliationInfo']:
                        affiliations_set.add(str(aff.get('Affiliation', '')))
                        
        authors_curto_str = "; ".join(authors_curto)
        authors_completo_str = "; ".join(authors_completo) # [NOVO] String de autores completos
        affiliations_str = "; ".join(list(affiliations_set))
        
        # 6. Agências Financiadoras
        grants_list = []
        if 'GrantList' in article_data:
            for grant in article_data['GrantList']:
                agency = grant.get('Agency')
                if agency:
                    grants_list.append(str(agency))
        grants_str = "; ".join(grants_list)
        
        # 7. Resumo
        abstract = ""
        if 'Abstract' in article_data and 'AbstractText' in article_data['Abstract']:
            abstract = " ".join([str(text) for text in article_data['Abstract']['AbstractText']])
            
        # 8. Termos MeSH
        mesh_terms_list = []
        if 'MeshHeadingList' in medline:
            for mesh_item in medline['MeshHeadingList']:
                descriptor = str(mesh_item['DescriptorName'])
                qualifiers = [str(q) for q in mesh_item.get('QualifierName', [])]
                
                if qualifiers:
                    for q in qualifiers:
                        mesh_terms_list.append(f"{descriptor}/{q}")
                else:
                    mesh_terms_list.append(descriptor)
                    
        mesh_string = "; ".join(mesh_terms_list)
        
        # ========================================================
        # EMPACOTANDO OS DADOS NO DICIONÁRIO
        # ========================================================
        data.append({
            'PMID': pmid,
            'DOI': doi,
            'Ano': str(year)[:4],
            'Revista': journal,
            'País_Revista': journal_country, # [NOVO]
            'Idioma': language_str,          # [NOVO]
            'Tipo_Publicacao': pub_types_str,
            'Titulo_Artigo': title,
            'Primeiro_Autor': first_author,
            'Autores_Curto': authors_curto_str,
            'Autores_Completos': authors_completo_str, # [NOVO]
            'Afiliacoes': affiliations_str,
            'Agencias_Financiadoras': grants_str,
            'Termos_MeSH': mesh_string,
            'Abstract': abstract
        })

    # Criando o DataFrame final
    df = pd.DataFrame(data)
    print("Extração concluída com sucesso!")
    
    # Exibindo as colunas novas
    display(df.head())
    
else:
    print("Nenhum dado para extrair. Certifique-se de que o download na Célula 1 funcionou.")

35827 artigos lidos de brasil_dtns_pubmed_bruto.csv.


,PMID,DOI,Ano,Revista,País_Revista,Idioma,Tipo_Publicacao,Titulo_Artigo,Primeiro_Autor,Autores_Curto,Autores_Completos,Afiliacoes,Agencias_Financiadoras,Termos_MeSH,Abstract
0,19869665,10.1084/jem.50.6.793,1929,The Journal of experimental medicine,United States,eng,Journal Article,STUDIES ON YELLOW FEVER IN SOUTH AMERICA : IV....,Davis NC,Davis NC; Shannon RC,"Davis, N C; Shannon, R C",Yellow Fever Laboratory of the International H...,,,1. Batches of Aëdes (Stegomyia) aegypti which ...
1,19870651,10.1084/jem.66.2.133,1937,The Journal of experimental medicine,United States,eng,Journal Article,THE MULTIPLICATION OF THE VIRUS OF YELLOW FEVE...,Whitman L,Whitman L,"Whitman, L",Laboratory of the Yellow Fever Service at Bahi...,,,Aëdes aegypti have been shown to be capable of...
2,45711,10.1590/s0074-02761976000200005,1976,Memorias do Instituto Oswaldo Cruz,Brazil,por,English Abstract; Journal Article; Research Su...,[Immunoglobulins in the chronic phase of Chaga...,De Freitas G,De Freitas G; Da Costa SC; Pereira Nde M; Quin...,"De Freitas, G; Da Costa, S C; Pereira, N de M;...","Instituto Oswaldo Cruz, Rio de Janeiro, Brasil.",,"Adolescent; Adult; Animals; Antibodies, Hetero...",Since divergent results were obtained by diffe...
3,400952,10.1016/0305-0491(77)90109-2,1977,"Comparative biochemistry and physiology. B, Co...",England,eng,"Journal Article; Research Support, Non-U.S. Gov't",Lipid composition of Trypanosoma cruzi.,Oliveira MM,Oliveira MM; Timm SL; Costa SC,"Oliveira, M M; Timm, S L; Costa, S C",Instituto de Biofisica de Universidade Federal...,,"Animals; Chromatography, Thin Layer; Host-Para...",1. Lipid composition of Trypanosoma cruzi epim...
4,400903,10.1016/0305-0491(77)90083-9,1977,"Comparative biochemistry and physiology. B, Co...",England,eng,"Journal Article; Research Support, Non-U.S. Gov't",Comparison of proteolytic enzymes from cercari...,Baba EH,Baba EH; Homewood CA; Gazzinelli G; Atkinson EM,"Baba, E H; Homewood, C A; Gazzinelli, G; Atkin...","Departamento de Bioquimica-Imunologia, U.F.M.G...",,"Animals; Electrophoresis, Polyacrylamide Gel; ...",


In [11]:
# Salvando os dados da PubMed antes do enriquecimento com a OpenAlex
if pubmed_lido_do_arquivo:
    print(f"Dados lidos de {ARQUIVO_PUBMED}; nada a salvar.")
elif pmids_falhos:
    # Se salvássemos agora, a próxima execução pularia o download com dados incompletos
    print("⚠️ Alguns PMIDs falharam no download; o arquivo não foi salvo. Rode o download de novo.")
else:
    df.to_csv(ARQUIVO_PUBMED, index=False, encoding='utf-8-sig')
    df.to_excel(ARQUIVO_PUBMED.replace('.csv', '.xlsx'), index=False)


Dados lidos de brasil_dtns_pubmed_bruto.csv; nada a salvar.


## Enriquecendo com informação da Openalex

In [12]:
final_lido_do_arquivo = all(os.path.exists(a) for a in (ARQUIVO_FINAL, ARQUIVO_AUTORIAS, ARQUIVO_INSTITUICOES))

if final_lido_do_arquivo:
    df_final = pd.read_csv(ARQUIVO_FINAL, dtype=str, keep_default_na=False)
    # Recuperando os tipos das colunas numéricas e de Sim/Não
    df_final['Citações_OpenAlex'] = pd.to_numeric(df_final['Citações_OpenAlex'], errors='coerce').astype('Int64')
    df_final['FWCI'] = pd.to_numeric(df_final['FWCI'], errors='coerce')
    for coluna in ['Encontrado_OpenAlex'] + [c for c in df_final.columns if c.startswith('Tem_Instituição_')]:
        df_final[coluna] = df_final[coluna] == 'True'
    print(f"{ARQUIVO_FINAL} já existe: enriquecimento com a OpenAlex pulado ({len(df_final)} artigos lidos).")
else:
    # 1. Pegamos a lista de PMIDs
    lista_pmids = df[df['PMID'] != '']['PMID'].tolist()

    #Numero de documentos por chamada de API
    tamanho_lote = 100

    # 2. Mantemos lotes de 25 para evitar URLs longas (Erro 400)
    def dividir_em_lotes(lista, tamanho=tamanho_lote):
        for i in range(0, len(lista), tamanho):
            yield lista[i:i + tamanho]

    lotes = list(dividir_em_lotes(lista_pmids))
    print(f"Buscando métricas na OpenAlex para {len(lista_pmids)} artigos (divididos em {len(lotes)} lotes)...")

    if api_key_openalex.strip() in ("", "COLE_SUA_CHAVE_AQUI"):
        raise ValueError("Defina a sua chave da OpenAlex em api_key_openalex (célula de configuração) e rode-a de novo.")
    api_key_openalex = api_key_openalex.strip()

    dados_openalex = []
    lotes_com_erro = 0
    autorias = []  # uma linha por artigo x instituição

    # 3. Fazendo as requisições
    for i, lote in enumerate(lotes):
        pmids_juntos = "|".join(lote)
        filtro_pmids = f"ids.pmid:{pmids_juntos}"
    
        url = f"https://api.openalex.org/works?filter={filtro_pmids}&per_page={tamanho_lote}&api_key={api_key_openalex}"
    
        try:
            # Até 3 tentativas para erros temporários do servidor (ex.: 504) ou de conexão
            for tentativa in range(3):
                try:
                    response = requests.get(url, timeout=60)
                    if response.status_code not in (429, 500, 502, 503, 504):
                        break
                except requests.exceptions.RequestException:
                    if tentativa == 2:
                        raise
                print(f"Lote {i+1}: tentativa {tentativa + 1} falhou, tentando de novo...")
                time.sleep(5 * (tentativa + 1))
        
            if response.status_code == 200:
                resultados = response.json().get('results', [])
            
                for work in resultados:
                    pmid_bruto = (work.get('ids') or {}).get('pmid') or ''
                    pmid_limpo = pmid_bruto.split('/')[-1] if pmid_bruto else ""
                
                    citacoes = work.get('cited_by_count', 0)

                    fwci = work.get('fwci') # Se não existir, assume 0.0
                
                    oa_data = work.get('open_access') or {}
                    oa_status = oa_data.get('oa_status', 'unknown')
                    is_oa = oa_data.get('is_oa', False)
                    acesso_aberto_str = "Sim" if is_oa else "Não"
                
                    # Campos ausentes vêm como None (null) na OpenAlex, por isso o "or {}"
                    publisher = ((work.get('primary_location') or {}).get('source') or {}).get('host_organization_name')
                    if publisher is None:
                        publisher = 'Desconhecido'
                
                    topic = work.get('primary_topic', {})
                    topic_name = topic.get('display_name', 'Sem Tópico') if topic else 'Sem Tópico'

                    # Países de todos os autores (útil para checar a cobertura do filtro de afiliação)
                    paises = sorted({p for a in work.get('authorships') or [] for p in a.get('countries') or []})

                    # Instituições de todos os autores (cada instituição conta uma vez por artigo)
                    instituicoes = {}
                    for autoria in work.get('authorships') or []:
                        for inst in autoria.get('institutions') or []:
                            if inst.get('id'):
                                instituicoes[inst['id']] = inst
                    for inst in instituicoes.values():
                        autorias.append({
                            'PMID': pmid_limpo,
                            'ID_Instituicao': inst['id'].split('/')[-1],
                            'Instituicao': inst.get('display_name'),
                            'Tipo_Instituicao': inst.get('type'),
                            'Pais_Instituicao': inst.get('country_code'),
                            # Instituição e suas "mães" (ex.: hospital universitário -> universidade)
                            'Lineage': "; ".join(l.split('/')[-1] for l in inst.get('lineage') or []),
                        })
                
                    dados_openalex.append({
                        'PMID': pmid_limpo,
                        'Citações_OpenAlex': citacoes,
                        'FWCI': fwci, 
                        'Status_OA': oa_status,
                        'Acesso_Aberto': acesso_aberto_str,
                        'Editora': publisher,
                        'Tópico_OpenAlex': topic_name,
                        'Países_Autores_OpenAlex': "; ".join(paises),
                        'Instituições_OpenAlex': "; ".join(sorted({i.get('display_name') or '' for i in instituicoes.values()}))
                    })
            
            else:
                # Este else agora está corretamente vinculado ao if da resposta 200
                print(f"Erro no lote {i+1}: {response.status_code} - {response.text[:300]}")
                lotes_com_erro += 1
                if response.status_code in (401, 403):
                    # Erro de autenticação: não adianta continuar com os outros lotes
                    raise RuntimeError("A OpenAlex recusou a chave de API. Confira o valor de api_key_openalex.")
            
        except RuntimeError:
            raise
        except Exception as e:
            print(f"Falha no lote {i+1}: {e}")
            lotes_com_erro += 1
        
        time.sleep(0.2)

    # 4. Criando e mesclando o DataFrame
    colunas_esperadas = ['PMID', 'Citações_OpenAlex', 'FWCI', 'Status_OA', 'Acesso_Aberto', 'Editora', 'Tópico_OpenAlex', 'Países_Autores_OpenAlex', 'Instituições_OpenAlex']
    df_openalex = pd.DataFrame(dados_openalex) if len(dados_openalex) > 0 else pd.DataFrame(columns=colunas_esperadas)

    print(f"Extração concluída! Foram encontrados dados para {len(df_openalex)} artigos na OpenAlex.")

    df_final = pd.merge(df, df_openalex, on='PMID', how='left')

    # Marca quais artigos foram encontrados na OpenAlex
    df_final['Encontrado_OpenAlex'] = df_final['PMID'].isin(df_openalex['PMID'])

    # Artigos não encontrados ficam sem valor (e não com 0 citações)
    df_final['Citações_OpenAlex'] = df_final['Citações_OpenAlex'].astype('Int64')
    df_final['FWCI'] = pd.to_numeric(df_final['FWCI'], errors='coerce')
    df_final['Status_OA'] = df_final['Status_OA'].fillna('unknown')
    df_final['Acesso_Aberto'] = df_final['Acesso_Aberto'].fillna('Desconhecido')
    df_final['Editora'] = df_final['Editora'].fillna('Desconhecido')
    df_final['Tópico_OpenAlex'] = df_final['Tópico_OpenAlex'].fillna('Sem Tópico')
    df_final['Países_Autores_OpenAlex'] = df_final['Países_Autores_OpenAlex'].fillna('')
    df_final['Instituições_OpenAlex'] = df_final['Instituições_OpenAlex'].fillna('')

    print(f"{df_final['Encontrado_OpenAlex'].sum()} de {len(df_final)} artigos encontrados na OpenAlex.")


Buscando métricas na OpenAlex para 35827 artigos (divididos em 359 lotes)...
Extração concluída! Foram encontrados dados para 35730 artigos na OpenAlex.
35730 de 35827 artigos encontrados na OpenAlex.


### Estado de cada instituição

O registro do artigo na OpenAlex traz as instituições dos autores, mas não o estado. Baixamos o cadastro de todas as instituições brasileiras da OpenAlex e descobrimos o estado de cada uma pela latitude/longitude, verificando em qual contorno estadual do IBGE o ponto cai. O campo `geo.region` da OpenAlex só está preenchido para cerca de 1/3 das instituições; onde existe, ele concordou 100% com o estado calculado pelas coordenadas.

In [13]:
if final_lido_do_arquivo:
    df_autorias = pd.read_csv(ARQUIVO_AUTORIAS, dtype=str, keep_default_na=False)
    df_autorias['Norte'] = df_autorias['Norte'] == 'True'
    df_instituicoes = pd.read_csv(ARQUIVO_INSTITUICOES, dtype={'ID_Instituicao': str})
    print(f"{ARQUIVO_AUTORIAS} já existe: busca de estados pulada.")
else:
    from shapely.geometry import shape, Point

    UFS_NORTE = {'AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'}
    REGIAO_POR_UF = {
        **dict.fromkeys(['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'], 'Norte'),
        **dict.fromkeys(['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'], 'Nordeste'),
        **dict.fromkeys(['DF', 'GO', 'MT', 'MS'], 'Centro-Oeste'),
        **dict.fromkeys(['ES', 'MG', 'RJ', 'SP'], 'Sudeste'),
        **dict.fromkeys(['PR', 'RS', 'SC'], 'Sul'),
    }

    # 1. Baixando o cadastro de todas as instituições brasileiras
    print("Baixando instituições brasileiras da OpenAlex...")
    instituicoes_br = []
    cursor = '*'
    while cursor:
        r = requests.get('https://api.openalex.org/institutions', params={
            'filter': 'country_code:BR', 'per_page': 200, 'cursor': cursor,
            'select': 'id,ror,display_name,display_name_acronyms,type,geo,lineage,works_count,cited_by_count',
            'api_key': api_key_openalex,
        }, timeout=60)
        r.raise_for_status()
        resposta = r.json()
        if not resposta['results']:
            break
        instituicoes_br += resposta['results']
        cursor = resposta['meta'].get('next_cursor')
        time.sleep(0.1)
    print(f"{len(instituicoes_br)} instituições brasileiras baixadas.")

    # 2. Contornos dos estados (IBGE)
    url_ibge = ("https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR"
                "?formato=application/vnd.geo+json&intrarregiao=UF&qualidade=intermediaria")
    estados_geo = requests.get(url_ibge, timeout=120).json()

    # Código IBGE de cada estado -> sigla (os estados do Norte são os que começam com 1)
    CODIGO_IBGE_UF = {
        '11': 'RO', '12': 'AC', '13': 'AM', '14': 'RR', '15': 'PA', '16': 'AP', '17': 'TO',
        '21': 'MA', '22': 'PI', '23': 'CE', '24': 'RN', '25': 'PB', '26': 'PE', '27': 'AL', '28': 'SE', '29': 'BA',
        '31': 'MG', '32': 'ES', '33': 'RJ', '35': 'SP',
        '41': 'PR', '42': 'SC', '43': 'RS',
        '50': 'MS', '51': 'MT', '52': 'GO', '53': 'DF',
    }
    poligonos = [(CODIGO_IBGE_UF[f['properties']['codarea']], shape(f['geometry']))
                 for f in estados_geo['features']]

    def uf_por_coordenada(lat, lon):
        if lat is None or lon is None:
            return None
        ponto = Point(lon, lat)  # atenção: a ordem é (longitude, latitude)
        for uf, poligono in poligonos:
            if poligono.contains(ponto):
                return uf
        # Pontos no litoral ou numa divisa podem cair fora por imprecisão do contorno:
        # usa o estado mais próximo, se estiver a menos de ~10 km
        uf, poligono = min(poligonos, key=lambda p: p[1].distance(ponto))
        return uf if poligono.distance(ponto) < 0.1 else None

    # 3. Tabela de instituições brasileiras
    linhas = []
    for inst in instituicoes_br:
        geo = inst.get('geo') or {}
        uf = uf_por_coordenada(geo.get('latitude'), geo.get('longitude'))
        linhas.append({
            'ID_Instituicao': inst['id'].split('/')[-1],
            'ROR': inst.get('ror'),
            'Instituicao': inst.get('display_name'),
            'Siglas': "; ".join(inst.get('display_name_acronyms') or []),
            'Tipo_Instituicao': inst.get('type'),
            'Cidade': geo.get('city'),
            'UF': uf,
            'Regiao': REGIAO_POR_UF.get(uf),
            'Norte': uf in UFS_NORTE,
            'Latitude': geo.get('latitude'),
            'Longitude': geo.get('longitude'),
            'Lineage': "; ".join(l.split('/')[-1] for l in inst.get('lineage') or []),
            'Producao_Total_OpenAlex': inst.get('works_count'),
            'Citacoes_Totais_OpenAlex': inst.get('cited_by_count'),
        })
    df_instituicoes = pd.DataFrame(linhas)
    print(f"Instituições do Norte: {df_instituicoes['Norte'].sum()}")
    sem_uf = df_instituicoes[df_instituicoes['UF'].isna()]
    print(f"Instituições sem estado: {len(sem_uf)}")
    display(sem_uf[['Instituicao', 'Cidade', 'Latitude', 'Longitude']])

    uf_por_instituicao = df_instituicoes.dropna(subset=['UF']).set_index('ID_Instituicao')['UF']

    # 4. Tabela de autorias (artigo x instituição) com UF
    colunas_autorias = ['PMID', 'ID_Instituicao', 'Instituicao', 'Tipo_Instituicao', 'Pais_Instituicao', 'Lineage']
    df_autorias = pd.DataFrame(autorias, columns=colunas_autorias)
    df_autorias['UF'] = df_autorias['ID_Instituicao'].map(uf_por_instituicao)
    df_autorias['Regiao'] = df_autorias['UF'].map(REGIAO_POR_UF)
    df_autorias['Norte'] = df_autorias['UF'].isin(UFS_NORTE)

    # 5. Levando os estados para a tabela de artigos
    juntar = lambda s: "; ".join(sorted(set(s)))
    estados = df_autorias.dropna(subset=['UF']).groupby('PMID')['UF'].agg(juntar)
    inst_norte = df_autorias[df_autorias['Norte']].groupby('PMID')['Instituicao'].agg(juntar)

    df_final['Estados_OpenAlex'] = df_final['PMID'].map(estados).fillna('')
    df_final['Instituições_Norte'] = df_final['PMID'].map(inst_norte).fillna('')
    df_final['Tem_Instituição_Norte'] = df_final['Instituições_Norte'] != ''

    # Regiões: uma coluna legível e uma coluna Sim/Não por região (para filtrar e contar)
    regioes = df_autorias.dropna(subset=['Regiao']).groupby('PMID')['Regiao'].agg(juntar)
    df_final['Regiões_OpenAlex'] = df_final['PMID'].map(regioes).fillna('')
    for regiao in ['Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']:
        df_final[f'Tem_Instituição_{regiao}'] = df_final['Regiões_OpenAlex'].str.contains(regiao)

    print("Artigos com pelo menos uma instituição de cada região:")
    for regiao in ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']:
        print(f"  {regiao}: {df_final[f'Tem_Instituição_{regiao}'].sum()}")

    # 6. Conferência: vínculos com instituição brasileira que ficaram sem estado
    br_sem_uf = df_autorias[(df_autorias['Pais_Instituicao'] == 'BR') & df_autorias['UF'].isna()]
    print(f"Vínculos com instituição brasileira sem estado identificado: {len(br_sem_uf)}")


Baixando instituições brasileiras da OpenAlex...
2073 instituições brasileiras baixadas.
Instituições do Norte: 74
Instituições sem estado: 1


,Instituicao,Cidade,Latitude,Longitude
418,Centro Universitário Euroamericano,Brasília,19.546736,-99.033119


Artigos com pelo menos uma instituição de cada região:
  Norte: 2317
  Nordeste: 7002
  Centro-Oeste: 4071
  Sudeste: 27913
  Sul: 3952
Vínculos com instituição brasileira sem estado identificado: 13


In [14]:
print("\n=== VISUALIZAÇÃO DO DATAFRAME ENRIQUECIDO ===")
display(df_final.head())


=== VISUALIZAÇÃO DO DATAFRAME ENRIQUECIDO ===


,PMID,DOI,Ano,Revista,País_Revista,Idioma,Tipo_Publicacao,Titulo_Artigo,Primeiro_Autor,Autores_Curto,...,Instituições_OpenAlex,Encontrado_OpenAlex,Estados_OpenAlex,Instituições_Norte,Tem_Instituição_Norte,Regiões_OpenAlex,Tem_Instituição_Nordeste,Tem_Instituição_Centro-Oeste,Tem_Instituição_Sudeste,Tem_Instituição_Sul
0,19869665,10.1084/jem.50.6.793,1929,The Journal of experimental medicine,United States,eng,Journal Article,STUDIES ON YELLOW FEVER IN SOUTH AMERICA : IV....,Davis NC,Davis NC; Shannon RC,...,Rockefeller Foundation,True,,,False,,False,False,False,False
1,19870651,10.1084/jem.66.2.133,1937,The Journal of experimental medicine,United States,eng,Journal Article,THE MULTIPLICATION OF THE VIRUS OF YELLOW FEVE...,Whitman L,Whitman L,...,,True,,,False,,False,False,False,False
2,45711,10.1590/s0074-02761976000200005,1976,Memorias do Instituto Oswaldo Cruz,Brazil,por,English Abstract; Journal Article; Research Su...,[Immunoglobulins in the chronic phase of Chaga...,De Freitas G,De Freitas G; Da Costa SC; Pereira Nde M; Quin...,...,Conselho Nacional de Desenvolvimento Científic...,True,DF; RJ,,False,Centro-Oeste; Sudeste,False,True,True,False
3,400952,10.1016/0305-0491(77)90109-2,1977,"Comparative biochemistry and physiology. B, Co...",England,eng,"Journal Article; Research Support, Non-U.S. Gov't",Lipid composition of Trypanosoma cruzi.,Oliveira MM,Oliveira MM; Timm SL; Costa SC,...,Fundação Oswaldo Cruz; Universidade Federal do...,True,RJ,,False,Sudeste,False,False,True,False
4,400903,10.1016/0305-0491(77)90083-9,1977,"Comparative biochemistry and physiology. B, Co...",England,eng,"Journal Article; Research Support, Non-U.S. Gov't",Comparison of proteolytic enzymes from cercari...,Baba EH,Baba EH; Homewood CA; Gazzinelli G; Atkinson EM,...,Liverpool School of Tropical Medicine; Univers...,True,MG,,False,Sudeste,False,False,True,False


## Salvando dataframe final em arquivo estático (csv e xlsx)

In [15]:
if final_lido_do_arquivo:
    print(f"Dados lidos de {ARQUIVO_FINAL}; nada a salvar.")
elif lotes_com_erro:
    # Se salvássemos agora, a próxima execução pularia o enriquecimento com dados incompletos
    print(f"⚠️ {lotes_com_erro} lotes da OpenAlex falharam; os arquivos não foram salvos. Rode o enriquecimento de novo.")
else:
    df_final.to_csv(ARQUIVO_FINAL, index=False, encoding='utf-8-sig')
    df_final.to_excel(ARQUIVO_FINAL.replace('.csv', '.xlsx'), index=False)

    # Tabela com uma linha por artigo x instituição (para contar por IES e por estado)
    df_autorias.to_csv(ARQUIVO_AUTORIAS, index=False, encoding='utf-8-sig')

    # Cadastro das instituições brasileiras, com estado
    df_instituicoes.to_csv(ARQUIVO_INSTITUICOES, index=False, encoding='utf-8-sig')


/tmp/ipykernel_44294/1111694965.py:8: UserWarning: Cell contents too long (33418), truncated to 32767 characters
  df_final.to_excel(ARQUIVO_FINAL.replace('.csv', '.xlsx'), index=False)
/tmp/ipykernel_44294/1111694965.py:8: UserWarning: Cell contents too long (51132), truncated to 32767 characters
  df_final.to_excel(ARQUIVO_FINAL.replace('.csv', '.xlsx'), index=False)
/tmp/ipykernel_44294/1111694965.py:8: UserWarning: Cell contents too long (34046), truncated to 32767 characters
  df_final.to_excel(ARQUIVO_FINAL.replace('.csv', '.xlsx'), index=False)
/tmp/ipykernel_44294/1111694965.py:8: UserWarning: Cell contents too long (49145), truncated to 32767 characters
  df_final.to_excel(ARQUIVO_FINAL.replace('.csv', '.xlsx'), index=False)
/tmp/ipykernel_44294/1111694965.py:8: UserWarning: Cell contents too long (52152), truncated to 32767 characters
  df_final.to_excel(ARQUIVO_FINAL.replace('.csv', '.xlsx'), index=False)
/tmp/ipykernel_44294/1111694965.py:8: UserWarning: Cell contents too l

## Análise exploratória dos dados

In [16]:
## Analisando os tipos de documentos do dataset

# 1. Filtramos apenas os artigos que possuem a coluna Tipo_Publicacao preenchida
pub_types_series = df_final[df_final['Tipo_Publicacao'] != ""]['Tipo_Publicacao']

# 2. Separamos os termos pelo ponto e vírgula
pub_types_lists = pub_types_series.str.split(';')

# 3. Usamos o 'explode' para separar cada tipo em uma linha e limpamos os espaços
pub_types_exploded = pub_types_lists.explode().str.strip()

# 4. Removemos textos vazios, caso existam
pub_types_exploded = pub_types_exploded[pub_types_exploded != ""]

# 5. Contamos a frequência de cada tipo de documento
pub_types_counts = pub_types_exploded.value_counts()

# 6. Criamos a tabela final
tabela_tipos = pd.DataFrame({
    'Total de Documentos': pub_types_counts,
    # Porcentagem calculada com base no total de artigos da base
    'Porcentagem na Base (%)': (pub_types_counts / len(df_final) * 100).round(2)
})

tabela_tipos.index.name = 'Tipo de Publicação'

print(f"Foram encontrados {len(tabela_tipos)} tipos de publicação únicos.")
print("\n=== DISTRIBUIÇÃO DOS TIPOS DE DOCUMENTOS ===")
display(tabela_tipos)

Foram encontrados 54 tipos de publicação únicos.

=== DISTRIBUIÇÃO DOS TIPOS DE DOCUMENTOS ===


,Total de Documentos,Porcentagem na Base (%)
Tipo de Publicação,,
Journal Article,35148,98.10
"Research Support, Non-U.S. Gov't",15934,44.47
Review,2722,7.60
Case Reports,1710,4.77
Comparative Study,1589,4.44
"Research Support, N.I.H., Extramural",1072,2.99
English Abstract,915,2.55
Evaluation Study,553,1.54
Systematic Review,441,1.23


Como a maioria esmagadora dos documentos são artigos, iremos usar os termos documentos, publicações e artigos de forma intercambiavel daqui pra frente.

In [17]:
## Analisando a porcentagem de artigos sem MeSH por ano (espera-se que seja maior para documentos mais recentes)

# 1. Criamos uma coluna temporária de Verdadeiro/Falso (True/False) 
# que será True se o artigo não tiver MeSH
df_final_analise = df_final.copy()
df_final_analise['Sem_MeSH'] = (df_final_analise['Termos_MeSH'] == "")

# 2. Agrupamos os dados por 'Ano'
tabela_anos = df_final_analise.groupby('Ano').agg(
    Total_Artigos=('PMID', 'count'),          # Conta quantos PMIDs tem naquele ano
    Artigos_Sem_MeSH=('Sem_MeSH', 'sum')      # Soma quantos são True (sem MeSH)
)

# 3. Calculamos a porcentagem matemática
tabela_anos['Porcentagem_Sem_MeSH (%)'] = (tabela_anos['Artigos_Sem_MeSH'] / tabela_anos['Total_Artigos'] * 100).round(1)

# 4. Ordenamos para ver os anos mais recentes primeiro
tabela_anos = tabela_anos.sort_index(ascending=False)

# ========================================================
# [NOVO] ADICIONANDO A LINHA DE TOTAIS
# ========================================================
soma_total_artigos = tabela_anos['Total_Artigos'].sum()
soma_artigos_sem_mesh = tabela_anos['Artigos_Sem_MeSH'].sum()

# Calcula a porcentagem global (evitando erro de divisão por zero caso o df_final esteja vazio)
if soma_total_artigos > 0:
    porcentagem_global = round((soma_artigos_sem_mesh / soma_total_artigos) * 100, 1)
else:
    porcentagem_global = 0.0

# Adiciona a nova linha usando a "chave" TOTAL no índice
tabela_anos.loc['TOTAL'] = [soma_total_artigos, soma_artigos_sem_mesh, porcentagem_global]

# Exibe a tabela completa
display(tabela_anos)

,Total_Artigos,Artigos_Sem_MeSH,Porcentagem_Sem_MeSH (%)
Ano,,,
2027,3.0,0.0,0.0
2026,1655.0,511.0,30.9
2025,1991.0,528.0,26.5
2024,1731.0,337.0,19.5
2023,1747.0,440.0,25.2
2022,1923.0,377.0,19.6
2021,2158.0,291.0,13.5
2020,2149.0,243.0,11.3
2019,1849.0,182.0,9.8


In [18]:
# -------------------------------------------------------------------
# IDIOMA DO ARTIGO
# -------------------------------------------------------------------
print("=== DISTRIBUIÇÃO POR IDIOMA ===")

# Criando a tabela de Idiomas
tabela_idiomas = pd.DataFrame({
    'Quantidade': df_final['Idioma'].value_counts(),
    'Porcentagem (%)': (df_final['Idioma'].value_counts(normalize=True) * 100).round(2)
})

# Renomeando o índice (que contém o nome do idioma) para ficar mais limpo
tabela_idiomas.index.name = 'Idioma'

display(tabela_idiomas)


# -------------------------------------------------------------------
# PAÍS DA REVISTA
# -------------------------------------------------------------------
print("\n=== DISTRIBUIÇÃO POR PAÍS DA REVISTA ===")

# Criando a tabela de Países
tabela_paises = pd.DataFrame({
    'Quantidade': df_final['País_Revista'].value_counts(),
    'Porcentagem (%)': (df_final['País_Revista'].value_counts(normalize=True) * 100).round(2)
})

tabela_paises.index.name = 'País da Revista'

# Como podem existir dezenas de países, vamos exibir os 15 primeiros
# (que geralmente representam 90%+ das publicações)
display(tabela_paises.head(15))

=== DISTRIBUIÇÃO POR IDIOMA ===


,Quantidade,Porcentagem (%)
Idioma,,
eng,34004,94.91
por,1122,3.13
eng; por,477,1.33
por; eng,85,0.24
spa,74,0.21
eng; por; spa,23,0.06
eng; spa,19,0.05
fre,11,0.03
por; spa,5,0.01



=== DISTRIBUIÇÃO POR PAÍS DA REVISTA ===


,Quantidade,Porcentagem (%)
País da Revista,,
United States,9228,25.76
England,7849,21.91
Brazil,7001,19.54
Netherlands,3997,11.16
Switzerland,3610,10.08
Germany,1697,4.74
France,541,1.51
Spain,337,0.94
United Arab Emirates,207,0.58


In [19]:
# -------------------------------------------------------------------
# TOTAL DE ARTIGOS CONTENDO CADA TERMO MESH
# -------------------------------------------------------------------

# 1. Filtramos apenas os artigos que possuem Termos MeSH (ignoramos os vazios)
mesh_series = df_final[df_final['Termos_MeSH'] != ""]['Termos_MeSH']

# 2. Separamos os termos pelo ponto e vírgula (vira uma lista de termos)
mesh_lists = mesh_series.str.split(';')

# 3. Usamos o 'explode' para separar cada termo em uma linha e 'strip' para limpar espaços
mesh_exploded = mesh_lists.explode().str.strip()

# 4. Removemos possíveis textos vazios gerados por acidentes de formatação no XML
mesh_exploded = mesh_exploded[mesh_exploded != ""]

# 5. Contamos a frequência de cada termo
mesh_counts = mesh_exploded.value_counts()

# 6. Criamos a tabela final
tabela_mesh = pd.DataFrame({
    'Total de Documentos': mesh_counts,
    # A porcentagem é calculada com base no total de artigos do DataFrame original (ou seja, conta artigos sem mesh)
    'Porcentagem na Base (%)': (mesh_counts / len(df_final) * 100).round(2) 
})

tabela_mesh.index.name = 'Termo MeSH'

print(f"Foram encontrados {len(tabela_mesh)} termos MeSH únicos.")
print("=== TOP TERMOS MESH MAIS FREQUENTES ===")

# Exibimos os 20 termos mais frequentes (você pode alterar esse número)
display(tabela_mesh.head(50))

Foram encontrados 41313 termos MeSH únicos.
=== TOP TERMOS MESH MAIS FREQUENTES ===


,Total de Documentos,Porcentagem na Base (%)
Termo MeSH,,
Humans,18719,52.25
Animals,18190,50.77
Female,10690,29.84
Male,9489,26.49
Adult,5513,15.39
Brazil/epidemiology,4914,13.72
Mice,4849,13.53
Middle Aged,4648,12.97
Brazil,4123,11.51


In [20]:
# Tabela de Acesso Aberto (Números e Proporções)
tabela_oa = pd.DataFrame({
    'Quantidade': df_final['Acesso_Aberto'].value_counts(),
    'Proporção (%)': (df_final['Acesso_Aberto'].value_counts(normalize=True) * 100).round(2)
})
tabela_oa.index.name = 'Status de Acesso'

print("=== DISTRIBUIÇÃO DE ACESSO ABERTO ===")
display(tabela_oa)

=== DISTRIBUIÇÃO DE ACESSO ABERTO ===


,Quantidade,Proporção (%)
Status de Acesso,,
Sim,24200,67.55
Não,11530,32.18
Desconhecido,97,0.27


In [21]:
# Análise de FWCI > 1
# Calculamos com base nos artigos que POSSUEM a métrica (ignorando NaN)
# FWCI > 1 significa que o artigo recebeu mais citações do que a média mundial (normalizado por campo e ano)
total_com_fwci = df_final['FWCI'].notna().sum()
artigos_fwci_alto = (df_final['FWCI'] > 1).sum()

porcentagem_fwci_alto = (artigos_fwci_alto / total_com_fwci * 100) if total_com_fwci > 0 else 0

# 3. Criando um resumo final
resumo_analise = pd.DataFrame({
    'Métrica': ['Total de Artigos', 'Artigos com FWCI disponível', 'Porcentagem com FWCI > 1 (%)'],
    'Valor': [len(df_final), total_com_fwci, round(porcentagem_fwci_alto, 2)]
})

print("\n=== INDICADOR DE IMPACTO (FWCI) ===")
display(resumo_analise)


=== INDICADOR DE IMPACTO (FWCI) ===


,Métrica,Valor
0,Total de Artigos,35827.00
1,Artigos com FWCI disponível,35354.00
2,Porcentagem com FWCI > 1 (%),58.06


## Quantas publicações a OpenAlex recuperaria?

Contagem na OpenAlex com a mesma lista de doenças (busca em título e resumo) e pelo menos um autor de instituição brasileira (`authorships.countries:BR`). Só depende das células de configuração e da estratégia de busca.

In [22]:
import re

# A OpenAlex separa filtros com vírgula, então removemos as vírgulas dos termos
# (ex.: "Leishmaniasis, Cutaneous" vira "Leishmaniasis Cutaneous")
termos_openalex = re.sub(r"\s+", " ", diseases_query.replace(",", " ")).strip()

def consultar_openalex(filtro_extra="", group_by=None):
    params = {
        "filter": f"title_and_abstract.search:{termos_openalex},authorships.countries:BR{filtro_extra}",
        "per_page": 200 if group_by else 1,
        "api_key": api_key_openalex,
    }
    if group_by:
        params["group_by"] = group_by
    r = requests.get("https://api.openalex.org/works", params=params)
    if r.status_code != 200:
        raise RuntimeError(f"Erro {r.status_code}: {r.text[:500]}")
    return r.json()

total_openalex = consultar_openalex()["meta"]["count"]
com_pmid = consultar_openalex(",has_pmid:true")["meta"]["count"]

print(f"Publicações na OpenAlex (Brasil + DTNs): {total_openalex}")
print(f"  com PMID (podem receber MeSH da PubMed): {com_pmid} ({com_pmid / total_openalex * 100:.1f}%)")
if "pmids" in globals():
    print(f"Para comparação, PubMed (Brazil[Affiliation]): {len(pmids)}")

# Distribuição por ano e por tipo de documento
por_ano = pd.DataFrame(consultar_openalex(group_by="publication_year")["group_by"])
por_ano = por_ano.rename(columns={"key": "Ano", "count": "Publicações"})[["Ano", "Publicações"]]
display(por_ano.sort_values("Ano", ascending=False))

por_tipo = pd.DataFrame(consultar_openalex(group_by="type")["group_by"])
por_tipo = por_tipo.rename(columns={"key_display_name": "Tipo", "count": "Publicações"})[["Tipo", "Publicações"]]
display(por_tipo)


Publicações na OpenAlex (Brasil + DTNs): 64179
  com PMID (podem receber MeSH da PubMed): 34418 (53.6%)


,Ano,Publicações
6,2026,3268
1,2025,3967
0,2024,4293
4,2023,3568
5,2022,3329
...,...,...
97,1930,1
96,1926,1
86,1923,2
85,1921,2


,Tipo,Publicações
0,article,57804
1,conference-paper,1258
2,book-chapter,1226
3,preprint,1070
4,review,569
5,dissertation,557
6,conference-abstract,453
7,other,288
8,dataset,240
9,editorial,206
